In [ ]:
import numpy as np

In [ ]:
# ==============================================================================
# FULL PIPELINE: 2D KINEMATIC MAP & DUAL-IMPACT DLA SKEWER EXTRACTION
# ==============================================================================
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import scipy.ndimage as ndimage
import scipy.stats as stats

import fake_spectra.spectra as spectra
import fake_spectra.abstractsnapshot as absn
import fake_spectra.unitsystem as unitsystem
import fake_spectra.gas_properties as gas_properties

# ------------------------------------------------------------------------------
# 1. STYLE CONFIGURATION & GLOBAL PARAMETERS
# ------------------------------------------------------------------------------
def matplotlib_default_config():
    """Returns the default dictionary configurations for publication-ready plots."""
    font = {'family' : 'serif', 'weight' : 'normal', 'size' : 14}
    rcnew = {
        "mathtext.fontset" : "cm", 
        "text.usetex": False, 
        'figure.frameon': True,
        'axes.linewidth': 2., 
        "axes.titlesize" : 20, 
        "axes.labelsize" : 20,
        "legend.fontsize" : 12, 
        'legend.fancybox': True, 
        'lines.linewidth': 2.5,
        'xtick.direction': 'in', 
        'ytick.direction': 'in',
        'xtick.top': True, 
        'ytick.right': True,
        'xtick.labelsize': 18, 
        'ytick.labelsize': 18
    }
    return font, rcnew

# Apply interactive backend and style
%matplotlib widget 
font, rcnew = matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

# Main Simulation Variables
BASE_PATH = '/home/STORAGE/MN5_resims/fiducial/hydro_output'
SNAP_NUM = "094"  

# Target Selection and Skewer Geometry
TARGET_HALO_ID = 410  # ID of the halo to analyze (must exist in the snapshot)
R_INNER = 0.1  # Impact parameter multiplier for the inner skewers
R_OUTER = 0.5  # Impact parameter multiplier for the outer skewers

SLICE_HALF_WIDTH = 0.1  # Native Mpc/h for the 2D map projection

# ------------------------------------------------------------------------------
# 2. MN5 COMPATIBILITY PATCHES
# ------------------------------------------------------------------------------
def safe_get_header_attr(self, attr):
    """Monkey patch to bypass MN5 missing high-word attributes and safe-load parameters."""
    if attr == "NumPart_Total_HighWord":
        return np.zeros(6, dtype=np.int32)
    try:
        val = self._f_handle["Header"].attrs[attr]
        if isinstance(val, np.ndarray) and np.size(val) == 1:
            try: return val[0]
            except IndexError: return val
        return val
    except KeyError:
        if "Parameters" in self._f_handle:
            try: return self._f_handle["Parameters"].attrs[attr]
            except KeyError: pass
        raise KeyError(f"Could not find attribute '{attr}'")

absn.AbstractSnapshot.get_header_attr = safe_get_header_attr

class MN5GasProperties(gas_properties.GasProperties):
    """Forces fake_spectra to load the ground truth Self-Shielding threshold from MN5."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        if self.absnap is not None:
            try: self.PhysDensThresh = self.absnap.get_header_attr("SelfShieldingDensity")
            except KeyError: self.PhysDensThresh = 0.1295
        else:
            self.PhysDensThresh = 0.1295

mn5_units = unitsystem.UnitSystem(
    UnitLength_in_cm=3.085678e24, UnitMass_in_g=1.989e43, UnitVelocity_in_cm_per_s=1e5
)

# ------------------------------------------------------------------------------
# 3. LOCATE TARGET HALO & LOAD GAS PARTICLES
# ------------------------------------------------------------------------------
print(f"Locating Halo {TARGET_HALO_ID}...")
group_dir = os.path.join(BASE_PATH, f"groups_{SNAP_NUM}")
file0 = os.path.join(group_dir, f"fof_subhalo_tab_{SNAP_NUM}.0.hdf5")

with h5py.File(file0, 'r') as f:
    num_group_files = f['Header'].attrs.get('NumFiles', f['Header'].attrs.get('NumFilesPerSnapshot', 1))
    box_size = f['Header'].attrs['BoxSize']
    redshift = f['Header'].attrs['Redshift']
    a_scale = 1.0 / (1.0 + redshift)

halo_pos, halo_r200, halo_mass, halo_vel = None, None, None, None
current_id_offset = 0

for i in range(num_group_files):
    filepath = os.path.join(group_dir, f"fof_subhalo_tab_{SNAP_NUM}.{i}.hdf5")
    if not os.path.exists(filepath): continue
    with h5py.File(filepath, 'r') as f:
        if 'Group' not in f or 'Group_M_Crit200' not in f['Group']: continue
        n_halos = len(f['Group']['Group_M_Crit200'])
        if TARGET_HALO_ID < current_id_offset + n_halos:
            local_idx = TARGET_HALO_ID - current_id_offset
            halo_pos = f['Group']['GroupPos'][local_idx]
            halo_r200 = f['Group']['Group_R_Crit200'][local_idx]
            halo_mass = f['Group']['Group_M_Crit200'][local_idx]
            halo_vel = f['Group']['GroupVel'][local_idx] 
            break
        current_id_offset += n_halos

if halo_pos is None: raise ValueError(f"Halo {TARGET_HALO_ID} not found!")
print(f"-> Found! R200: {halo_r200:.4f} Mpc/h, Mass: {halo_mass:.2e}, z: {redshift:.2f}")

print("Extracting Neutral Hydrogen and Velocities within 4xR200 box...")
snap_file_0 = os.path.join(BASE_PATH, f'snapdir_{SNAP_NUM}', f'snapshot_{SNAP_NUM}.0.hdf5')
with h5py.File(snap_file_0, 'r') as f:
    num_snap_files = f['Header'].attrs.get('NumFilesPerSnapshot', f['Header'].attrs.get('NumFiles', 1))

map_x, map_y, map_mass_HI, map_vx, map_vy = [], [], [], [], []
box_radius = 1.5 * halo_r200  

halo_v_pec_x = halo_vel[0] / a_scale
halo_v_pec_y = halo_vel[1] / a_scale

for i in range(num_snap_files):
    snap_file = os.path.join(BASE_PATH, f'snapdir_{SNAP_NUM}', f'snapshot_{SNAP_NUM}.{i}.hdf5')
    if not os.path.exists(snap_file): continue
    
    with h5py.File(snap_file, 'r') as f:
        if 'PartType0' not in f: continue
        coords = f['PartType0']['Coordinates'][:]
        masses = f['PartType0']['Masses'][:]
        vels = f['PartType0']['Velocities'][:] 
        
        nha = f['PartType0']['NeutralHydrogenAbundance'][:] if 'NeutralHydrogenAbundance' in f['PartType0'] else np.ones(len(masses)) * 0.76 
        
        dx = (coords[:, 0] - halo_pos[0] + box_size/2.0) % box_size - box_size/2.0
        dy = (coords[:, 1] - halo_pos[1] + box_size/2.0) % box_size - box_size/2.0
        dz = (coords[:, 2] - halo_pos[2] + box_size/2.0) % box_size - box_size/2.0
        
        mask = (np.abs(dz) <= SLICE_HALF_WIDTH) & (np.abs(dx) <= box_radius) & (np.abs(dy) <= box_radius)
        
        if np.sum(mask) > 0:
            map_x.append(dx[mask])
            map_y.append(dy[mask])
            map_mass_HI.append(masses[mask] * nha[mask])
            map_vx.append((vels[:, 0][mask] * np.sqrt(a_scale)) - halo_v_pec_x)
            map_vy.append((vels[:, 1][mask] * np.sqrt(a_scale)) - halo_v_pec_y)

x_gas, y_gas = np.concatenate(map_x), np.concatenate(map_y)
mass_HI_gas = np.concatenate(map_mass_HI)
vx_gas, vy_gas = np.concatenate(map_vx), np.concatenate(map_vy)

# ------------------------------------------------------------------------------
# 4. SKEWER GEOMETRY & NATIVE EXTRACTION
# ------------------------------------------------------------------------------
cofm_list, axis_list, radius_labels = [], [], []

for r_mult in [R_INNER, R_OUTER]:
    impact_code = r_mult * halo_r200
    offsets_2d = [(impact_code, 0), (-impact_code, 0), (0, impact_code), (0, -impact_code)]
    skewers_meta = []
    
    for off_y, off_z in offsets_2d: skewers_meta.append({'axis': 1, 'offset': np.array([0.0, off_y, off_z])}) 
    for off_x, off_z in offsets_2d: skewers_meta.append({'axis': 2, 'offset': np.array([off_x, 0.0, off_z])}) 
    for off_x, off_y in offsets_2d: skewers_meta.append({'axis': 3, 'offset': np.array([off_x, off_y, 0.0])}) 
    
    for sk in skewers_meta:
        new_pos = halo_pos + sk['offset']
        for k in range(3):
            if new_pos[k] < 0: new_pos[k] += box_size
            if new_pos[k] >= box_size: new_pos[k] -= box_size
        cofm_list.append(new_pos)
        axis_list.append(sk['axis'])
        radius_labels.append(r_mult)

cofm_array = np.array(cofm_list, dtype=np.float64)
axis_array = np.array(axis_list, dtype=np.int32)
radius_labels = np.array(radius_labels)

print(f"--> Extracting 24 skewers in RAM (DLA Mode)...")
save_file = f"temp_24skewers_halo{TARGET_HALO_ID}.hdf5"
ss_batch = spectra.Spectra(
    SNAP_NUM, BASE_PATH, cofm=cofm_array, axis=axis_array,
    res=10.0, savefile=save_file, reload_file=True,
    units=mn5_units, gasprop=MN5GasProperties, sf_neutral=True,
    filter_mode="DLAs", dla_phys_thresh="auto"
)
tau_batch = ss_batch.get_tau("H", 1, 1215)
flux_batch = np.exp(-tau_batch)
if os.path.exists(save_file): os.remove(save_file)

# ------------------------------------------------------------------------------
# 5. KINEMATIC CENTERING & ABSORPTION ALIGNMENT
# ------------------------------------------------------------------------------
def align_to_absorption_center(vel_array, flux_array):
    """Isolates the deepest absorption basin to align the spectrum."""
    threshold = np.min(flux_array) + 0.02 
    bottom_indices = np.where(flux_array <= threshold)[0]
    if len(bottom_indices) == 0: return vel_array, 0.0
    jumps = np.where(np.diff(bottom_indices) > 1)[0]
    
    if len(jumps) == 0:
        v_center = np.mean(vel_array[bottom_indices])
    else:
        starts = np.insert(bottom_indices[jumps + 1], 0, bottom_indices[0])
        ends = np.append(bottom_indices[jumps], bottom_indices[-1])
        best_center, min_dist = 0.0, np.inf
        for s, e in zip(starts, ends):
            island_v_center = np.mean(vel_array[np.arange(s, e + 1)])
            if np.abs(island_v_center) < min_dist:
                min_dist = np.abs(island_v_center)
                best_center = island_v_center
        v_center = best_center
    return vel_array - v_center, v_center

nbins, vmax = ss_batch.nbins, ss_batch.vmax
v_common = np.linspace(-vmax / 2.0, vmax / 2.0, nbins)
aligned_fluxes_inner, aligned_fluxes_outer = [], []

for i in range(24):
    ax_idx = axis_array[i] - 1
    v_cosmo = (halo_pos[ax_idx] / box_size) * vmax
    v_pec = halo_vel[ax_idx] * np.sqrt(a_scale)
    v_sys = v_cosmo + v_pec
    
    v_axis = np.arange(nbins) * (vmax / nbins)
    v_rel_wrapped = (v_axis - v_sys + vmax / 2.0) % vmax - vmax / 2.0
    sort_idx = np.argsort(v_rel_wrapped)
    
    flux_halo_centered = np.interp(v_common, v_rel_wrapped[sort_idx], flux_batch[i, sort_idx])
    v_aligned, _ = align_to_absorption_center(v_common, flux_halo_centered)
    flux_final = np.interp(v_common, v_aligned, flux_halo_centered)
    
    if np.isclose(radius_labels[i], R_INNER): aligned_fluxes_inner.append(flux_final)
    else: aligned_fluxes_outer.append(flux_final)

aligned_fluxes_inner, aligned_fluxes_outer = np.array(aligned_fluxes_inner), np.array(aligned_fluxes_outer)

mean_inner, std_inner = np.mean(aligned_fluxes_inner, axis=0), np.std(aligned_fluxes_inner, axis=0)
mean_outer, std_outer = np.mean(aligned_fluxes_outer, axis=0), np.std(aligned_fluxes_outer, axis=0)

in_up, in_low = np.clip(mean_inner + std_inner, 0, 1), np.clip(mean_inner - std_inner, 0, 1)
out_up, out_low = np.clip(mean_outer + std_outer, 0, 1), np.clip(mean_outer - std_outer, 0, 1)


In [ ]:
# ------------------------------------------------------------------------------
# 6. PAPER-READY MULTIPANEL PLOT
# ------------------------------------------------------------------------------
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(14, 15), gridspec_kw={'height_ratios': [1.2, 1]})

# --- TOP PANEL: 2D HI Map & Kinematics ---
ax_top.set_facecolor('black')
bins_res = 200
H, xedges, yedges = np.histogram2d(x_gas, y_gas, bins=bins_res, range=[[-box_radius, box_radius], [-box_radius, box_radius]], weights=mass_HI_gas)
H_smoothed = np.where(ndimage.gaussian_filter(H, sigma=6) <= 0, 1e-12, ndimage.gaussian_filter(H, sigma=6))
X, Y = np.meshgrid(xedges, yedges)

img = ax_top.pcolormesh(X, Y, H_smoothed.T, cmap='inferno', norm=LogNorm(vmin=np.percentile(H_smoothed[H_smoothed > 1e-12], 5), vmax=np.max(H_smoothed)))

# Quiver
stat_vx, x_edge_v, y_edge_v, _ = stats.binned_statistic_2d(x_gas, y_gas, vx_gas, statistic='mean', bins=35, range=[[-box_radius, box_radius], [-box_radius, box_radius]])
stat_vy, _, _, _ = stats.binned_statistic_2d(x_gas, y_gas, vy_gas, statistic='mean', bins=35, range=[[-box_radius, box_radius], [-box_radius, box_radius]])
X_v, Y_v = np.meshgrid((x_edge_v[:-1] + x_edge_v[1:])/2, (y_edge_v[:-1] + y_edge_v[1:])/2)
ax_top.quiver(X_v, Y_v, stat_vx.T, stat_vy.T, color='cyan', scale=4000, width=0.003, alpha=0.45, zorder=3)

# Legend Box & Quiver Key
key_box = patches.FancyBboxPatch((0.70, 0.905), 0.27, 0.065, boxstyle="round,pad=0.02", facecolor='black', edgecolor='white', alpha=0.8, transform=ax_top.transAxes, zorder=4)
ax_top.add_patch(key_box)
ax_top.quiverkey(ax_top.quiver([0], [0], [0], [0], color='cyan', scale=4000, width=0.003, alpha=1.0), X=0.88, Y=0.94, U=100, label='100 km/s', labelpos='W', color='cyan', labelcolor='white', fontproperties={'size': 14}, zorder=5)

# Geometry Circles
ax_top.add_patch(patches.Circle((0, 0), radius=R_INNER * halo_r200, linewidth=2.0, edgecolor='red', facecolor='none', linestyle='--', label=rf'Inner ($b = {R_INNER}\,R_{{200}}$)', zorder=6))
ax_top.add_patch(patches.Circle((0, 0), radius=halo_r200, linewidth=2.5, edgecolor='white', facecolor='none', linestyle='--', label=rf'$R_{{200}}$', zorder=6))
ax_top.add_patch(patches.Circle((0, 0), radius=R_OUTER * halo_r200, linewidth=2.0, edgecolor='green', facecolor='none', linestyle='--', label=rf'Outer ($b = {R_OUTER}\,R_{{200}}$)', zorder=6))

# Skewer Lines (Crosshairs)
for offset in [-R_INNER * halo_r200, R_INNER * halo_r200]:
    ax_top.axhline(offset, color='red', linestyle='-', alpha=0.5, lw=1.5)
    ax_top.axvline(offset, color='red', linestyle='-', alpha=0.5, lw=1.5)
for offset in [-R_OUTER * halo_r200, R_OUTER * halo_r200]:
    ax_top.axhline(offset, color='forestgreen', linestyle='-', alpha=0.5, lw=1.5)
    ax_top.axvline(offset, color='forestgreen', linestyle='-', alpha=0.5, lw=1.5)

ax_top.set_xlim(-box_radius, box_radius); ax_top.set_ylim(-box_radius, box_radius)
ax_top.set_xlabel(r'$\Delta X \ [\mathrm{Mpc}/h]$'); ax_top.set_ylabel(r'$\Delta Y \ [\mathrm{Mpc}/h]$')
ax_top.set_title(rf'Halo {TARGET_HALO_ID} Kinematics & HI Density ($z={redshift:.0f}$)')
ax_top.legend(loc='lower left', facecolor='black', edgecolor='white', labelcolor='white', framealpha=0.8, handlelength=1)

divider = make_axes_locatable(ax_top)
cbar = fig.colorbar(img, cax=divider.append_axes("right", size="4%", pad=0.1))
cbar.set_label(r'Smoothed HI Mass (Projected)')

# --- BOTTOM PANEL: Absorption Profiles ---
ax_bot.fill_between(v_common, out_low, out_up, color='forestgreen', alpha=0.15, edgecolor='none', label=rf'$1\sigma$ Dispersion (${R_OUTER}\,R_{{200}}$)')
ax_bot.fill_between(v_common, in_low, in_up, color='red', alpha=0.15, edgecolor='none', label=rf'$1\sigma$ Dispersion (${R_INNER}\,R_{{200}}$)')

for i in range(12):
    ax_bot.plot(v_common, aligned_fluxes_outer[i], color='forestgreen', alpha=0.2, lw=0.8, label=rf'Individual Skewers (${R_OUTER}\,R_{{200}}$)' if i==0 else "")
    ax_bot.plot(v_common, aligned_fluxes_inner[i], color='red', alpha=0.2, lw=0.8, label=rf'Individual Skewers (${R_INNER}\,R_{{200}}$)' if i==0 else "")

ax_bot.plot(v_common, mean_outer, color='forestgreen', alpha=1.0, lw=2.5, label=rf'Mean Flux (${R_OUTER}\,R_{{200}}$)')
ax_bot.plot(v_common, mean_inner, color='red', alpha=1.0, lw=2.5, label=rf'Mean Flux (${R_INNER}\,R_{{200}}$)')
ax_bot.axvline(0.0, color='black', linestyle='--', lw=1.5, alpha=0.7, label='Absorption Center')

ax_bot.set_xlim(-vmax / 2.0, vmax / 2.0); ax_bot.set_ylim(-0.05, 1.05)
ax_bot.set_xlabel(r'Relative Velocity $\Delta v$ [km/s]'); ax_bot.set_ylabel(r'Transmitted Flux $F/F_0$')
ax_bot.set_title(rf'Radial DLA Absorption Profiles')
ax_bot.grid(True, alpha=0.3, linestyle='--')

# INCREASED LEGEND SIZE & REPOSITIONED TO THE LEFT
ax_bot.legend(loc='lower left', framealpha=0.9, fontsize=14, ncol=1)

plt.tight_layout()

# Save as PDF
pdf_filename = f"Halo_{TARGET_HALO_ID}_RadialMapAndSpectra.pdf"
plt.savefig(pdf_filename, format='pdf', dpi=300, bbox_inches='tight')
print(f"\n[✔] Figure saved successfully as '{pdf_filename}'")
plt.show()